# Baseline Sales Model

**Stage 1, Step 4** — the first analytical capability, and the one every later step measures against.

This notebook walks the model's behaviour rather than re-deriving it. Training happens in `scripts/train_baseline.py`; here we load what it produced and interrogate it.

The question the model answers:

> **What would sales have been under normal conditions** — no promotion running, stock available?

That is a *counterfactual*, not a forecast. A forecast predicts what will happen **including** planned interventions; a baseline estimates what would have happened **without** them. Same machinery, opposite counterfactual, and conflating the two is the most common way uplift analysis goes wrong — a forecast that includes the promotion measures it against itself and reports roughly zero uplift.

---

## Prerequisite

```powershell
uv run python scripts/train_baseline.py --profile dev --seed 42
```

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1]))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from app.services.container import Container
from ml.baseline.evaluation import compute_metrics, irreducible_error
from ml.baseline.model import FittedBaselineModel
from ml.baseline.pipeline import build_panel, load_latent_demand

repository = Container().data_repository
MODEL_DIR = Path.cwd().parents[1] / "data" / "local" / "models" / "baseline"
print(f"model artifacts: {MODEL_DIR}")

## 1. The trained model

Six candidates were compared — three estimators × two promotion approaches — and one was selected from the numbers. Let's see which, and why.

In [ ]:
model = FittedBaselineModel.load_from(MODEL_DIR, repository)

print(f"estimator        : {model.estimator.name}")
print(f"version          : {model.version}")
print(f"promo approach   : {model.approach.value}")
print(f"features         : {len(model.feature_names)}")
print(f"interval         : {model.calibration.nominal_coverage:.0%} nominal" if model.calibration else "no interval")

In [ ]:
report_path = MODEL_DIR / "evaluation_report.md"
if report_path.is_file():
    print(report_path.read_text(encoding="utf-8"))
else:
    print("no evaluation report — run scripts/train_baseline.py first")

## 2. Reading the accuracy figure

A bare WMAPE is uninterpretable, and this is worth dwelling on because the headline number looks bad at first glance.

Step 2 stored `mean_demand` — the **true conditional mean** — alongside the realised count drawn from it. The error between those two is pure negative-binomial sampling noise: there is nothing left in it to learn. That is the floor no model can go below.

Compare the model against the floor rather than against zero.

In [ ]:
latent = load_latent_demand(repository)
floor = irreducible_error(latent)

print("Noise floor — the best any model could possibly do")
print(f"  {floor.summary()}")
print()
print("Demand is over-dispersed: with a mean near 60 and a dispersion parameter")
print("of 3–9, the coefficient of variation is large. A model at ~40% WMAPE is")
print(f"therefore at roughly {0.40 / floor.wmape:.2f}x the theoretical best — not 'inaccurate'.")
print()
print("A model scoring 20% here would be impossible on honest features,")
print("and should be read as evidence of leakage rather than of skill.")

## 3. The stockout test — the headline result

This is the validation that makes the step more than a fitted model.

Observed sales during a stockout measure **availability, not demand**. A model trained carelessly learns that a supply failure predicts low demand — exactly backwards — and normally that is *undetectable*, because such a model scores **better** against observed sales than the correct one does. It reproduces the censoring too.

Here the true uncensored demand was retained, so we can check what the model actually measured.

In [ ]:
panel = build_panel(repository, sample_pairs=400, seed=42)

test_start = model.metadata.get("test_start") if hasattr(model, "metadata") else None
dates = pd.to_datetime(panel["date"]).dt.date
recent = panel[dates >= sorted(dates.unique())[-120]]

scored = recent[["date", "product_id", "store_id", "units", "promotion_flag", "stockout_flag"]].copy()
scored["baseline_units"] = model.estimator.predict(recent[list(model.feature_names)])

truth = latent.merge(scored, on=["date", "product_id", "store_id"], how="inner")
print(f"{len(truth):,} scored rows with ground truth")

In [ ]:
stockouts = truth[truth["lost_units"] > 0]

vs_observed = stockouts["baseline_units"].sum() / stockouts["observed_units"].sum()
vs_latent = stockouts["baseline_units"].sum() / stockouts["latent_units"].sum()

print(f"stockout rows                    : {len(stockouts):,}")
print(f"baseline / observed (censored)   : {vs_observed:.2f}x   <- must be well above 1")
print(f"baseline / latent (true demand)  : {vs_latent:.2f}x")
print()
print("PASS — the model sees through the censoring" if vs_observed > 1.2 else "FAIL — the model learned the supply failure")

### Why the second ratio is below 1, and why that is correct

The intuitive test — "baseline should equal latent demand during stockouts" — is **wrong here**, and an earlier revision of this step failed because of it.

Stockouts in this data are **endogenous**: they happen *because* demand spiked. So stockout periods are not ordinary periods, and a baseline correctly predicting *normal* demand should land below the elevated latent demand of a stockout day.

Let's measure that elevation directly rather than assert it.

In [ ]:
so = latent[latent["lost_units"] > 0]["latent_units"].mean()
ok = latent[latent["lost_units"] == 0]["latent_units"].mean()

print(f"latent demand during stockouts : {so:.1f}")
print(f"latent demand otherwise        : {ok:.1f}")
print(f"elevation                      : {so / ok:.2f}x")
print()
print(f"So a correct baseline lands near 1/{so / ok:.2f} = {ok / so:.2f} of stockout-period")
print("latent demand. Judging it against 1.0 would disqualify a good model —")
print("which is exactly what happened before the criterion was corrected to")
print("use the ratio against *observed* sales, which has no such confound.")

## 4. Lost sales — the business question

"We stocked out last week. How much did we actually lose?"

The raw sales data cannot answer this: sales fell, but demand did not. Only a model that estimates demand can separate the two.

In [ ]:
estimated_loss = (stockouts["baseline_units"] - stockouts["observed_units"]).clip(lower=0).sum()
true_loss = stockouts["lost_units"].sum()

print(f"estimated lost units : {estimated_loss:>12,.0f}")
print(f"true lost units      : {true_loss:>12,.0f}")
print(f"error                : {(estimated_loss / true_loss - 1):>+12.1%}")
print()
print("Direction alone is not enough — an estimate 3x too large would still")
print("drive the wrong safety-stock decision.")

## 5. Promotions — and the bias that matters most

Two failure modes, pulling in opposite directions:

- **Baseline too high** → real uplift is hidden, effective promotions look like failures.
- **Baseline too low** → *phantom* uplift, every promotion looks profitable.

The second is far more dangerous. It produces good news, so nobody investigates, and budget flows toward promotions that do nothing.

The check: on days with **no promotion**, actual and baseline should agree. Any systematic gap there is phantom uplift waiting to be attributed.

In [ ]:
in_stock = truth[truth["lost_units"] == 0]
clean = in_stock[~in_stock["promotion_flag"].astype(bool)]
promoted = in_stock[in_stock["promotion_flag"].astype(bool)]

phantom = clean["units"].sum() / clean["baseline_units"].sum() - 1
measured = promoted["units"].sum() / promoted["baseline_units"].sum() - 1

print(f"non-promotional days — apparent uplift : {phantom:+.1%}   <- should be near zero")
print(f"promotional days     — measured uplift : {measured:+.1%}")
print()
print(f"signal-to-noise: real uplift is {abs(measured / phantom):.1f}x the phantom baseline drift" if phantom else "")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

daily = truth.groupby("date").agg(
    actual=("units", "sum"),
    baseline=("baseline_units", "sum"),
    promo_share=("promotion_flag", "mean"),
)

ax.plot(daily.index, daily["actual"], label="actual", linewidth=1.2)
ax.plot(daily.index, daily["baseline"], label="baseline", linewidth=1.2, linestyle="--")
ax.fill_between(
    daily.index, daily["actual"], daily["baseline"],
    where=daily["actual"] > daily["baseline"], alpha=0.2, label="gap (actual > baseline)",
)
ax.set_title("Actual vs baseline — the shaded area is what uplift analysis measures")
ax.set_ylabel("units")
ax.legend()
plt.tight_layout()
plt.show()

**The shaded area is not uplift.** It is a *difference*. Attributing it to promotions requires causal assumptions this model does not test — seasonality, competitor moves and price changes all live in that gap too. That caveat travels with every response the service returns, and it is the one an agent is most likely to drop.

## 6. Prediction intervals — a number that is earned

A confidence figure is the easiest thing in the system to fabricate. `confidence: 0.92` costs nothing to emit and means nothing.

Split conformal earns it: distribution-free, with a finite-sample guarantee. But the property that actually matters is that **achieved coverage is measured on held-out data and reported whatever it turns out to be**. A 90% interval covering 71% is a finding, not a detail to smooth over.

In [ ]:
if model.calibration is not None:
    from ml.baseline.conformal import measure_coverage

    report = measure_coverage(clean["units"], clean["baseline_units"], model.calibration)
    print(report.summary())
    print()
    print("calibrated" if report.is_calibrated else "NOT calibrated — the shortfall is reported rather than hidden")
else:
    print("this model carries no calibration")

`is_significant` therefore means something precise: **the gap falls outside the interval**, so it exceeds the model's normal error. That is the test Step 17's root-cause agent needs before calling a decline real rather than noise — and it is only meaningful because the coverage above was measured.

## 7. What drives the baseline

In [ ]:
importance = model.estimator.feature_importance()

if importance is not None and not importance.empty:
    top = importance.head(15).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(top["feature"], top["importance"])
    ax.set_title("Top features")
    plt.tight_layout()
    plt.show()
else:
    print("this estimator reports no feature importance")

SHAP is deliberately absent. At panel scale it costs considerably more than it adds for the question "what drives baseline demand", and permutation importance — the WMAPE degradation when a feature is shuffled — answers it directly. Recorded as a decision, not an omission.

## 8. Using it through the service

The service is the seam every later step goes through. Note that expected failures come back as **values, not exceptions** — by Step 16 a supervisor agent has to re-plan around them, and it can only do that with a failure it can read.

In [ ]:
from datetime import date

from app.schemas.baseline import BaselineRequest
from app.services.baseline_service import BaselineSalesService

service = BaselineSalesService(repository, model_dir=MODEL_DIR)
response = service.predict(
    BaselineRequest(start_date=date(2025, 10, 1), end_date=date(2025, 10, 14), include_records=True, max_records=5)
)

print(f"baseline units : {response.result.baseline_units:,.0f}")
print(f"actual units   : {response.result.actual_units:,.0f}")
print(f"rows           : {response.record_count:,}")
print(f"cold-start rows: {response.fallback_rows:,}")

In [ ]:
print("ASSUMPTIONS\n")
for item in response.assumptions:
    print(f"  - {item}")

print("\nWARNINGS\n")
for item in response.warnings or ["(none)"]:
    print(f"  - {item}")

In [ ]:
# A failure the caller can act on, rather than a stack trace.
bad = service.predict(BaselineRequest(start_date=date(2025, 6, 1), end_date=date(2025, 1, 1)))

print(f"error_code  : {bad.error_code}")
print(f"recoverable : {bad.recoverable}")
print(f"message     : {bad.message}")

---

## Summary

| Check | Why it matters |
|---|---|
| Accuracy vs **noise floor** | A bare WMAPE is uninterpretable; the ratio is the real statement |
| **Stockout ratio vs observed** | Proves the model learned demand, not the supply failure |
| **Phantom uplift ≈ 0** | A baseline biased low manufactures uplift on every promotion |
| **Measured** interval coverage | Separates a real interval from a fabricated confidence score |
| Cold-start rows flagged | A caller must tell an estimate from a guess |

### The limit worth restating

Exact baseline reconstruction on promotional rows is **not possible** from stored ground truth — the store-level `_promo_responsiveness` multiplier is latent and deliberately unpublished. Promotional validation is directional only.

**Next:** Step 5 builds promotion uplift on top of this baseline.